# Infant Brain Processing Pipeline - Step-by-Step Visualization

This notebook walks through the entire infant brain processing pipeline, visualizing each step.

**Pipeline**: Simplified iFS-only (no iBEAT2, no MATLAB)

**Your Data**:
- Subject: sub-01
- Session: ses-03
- Input: T1w MRI image

## Table of Contents
1. Setup and Configuration
2. Input Data Inspection
3. FreeSurfer Initial Processing
4. Infant FreeSurfer Processing
5. Label Remapping (iFS → FS compatibility)
6. White Matter Generation
7. Surface Reconstruction
8. Final Outputs and Quality Control

## 1. Setup and Configuration

In [ ]:
# Import libraries
import os
import json
import subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import plotting, image
from IPython.display import display, HTML, Image as IPImage
import pandas as pd
from pathlib import Path

# Set plotting parameters
plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")

In [ ]:
# Define paths
INPUT_T1W = "/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz"
INPUT_JSON = "/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.json"

# FreeSurfer directories
SUBJECTS_DIR = "/data02/share/bin-wu/data/human/brain/harvard_mri/processed/freesurfer"
SUBJECT_ID = "sub-01_ses-03"

# Create directories if needed
os.makedirs(SUBJECTS_DIR, exist_ok=True)

# Verify input files exist
print(f"Input T1w exists: {os.path.exists(INPUT_T1W)}")
print(f"Input JSON exists: {os.path.exists(INPUT_JSON)}")
print(f"\nSubjects directory: {SUBJECTS_DIR}")
print(f"Subject ID: {SUBJECT_ID}")

In [ ]:
# Set FreeSurfer environment (adjust paths for your system)
os.environ['FREESURFER_HOME'] = '/usr/local/freesurfer'  # Adjust this path
os.environ['SUBJECTS_DIR'] = SUBJECTS_DIR

# Verify FreeSurfer is available
try:
    result = subprocess.run(['which', 'recon-all'], capture_output=True, text=True)
    print(f"FreeSurfer found: {result.stdout.strip()}")
except:
    print("⚠️ FreeSurfer not found in PATH. Please source SetUpFreeSurfer.sh first.")

## 2. Input Data Inspection

In [ ]:
# Load and display JSON metadata
with open(INPUT_JSON, 'r') as f:
    metadata = json.load(f)

print("=" * 60)
print("MRI ACQUISITION PARAMETERS")
print("=" * 60)
for key, value in sorted(metadata.items()):
    print(f"{key:30s}: {value}")

In [ ]:
# Load T1w image
t1w_img = nib.load(INPUT_T1W)
t1w_data = t1w_img.get_fdata()

print("=" * 60)
print("T1W IMAGE PROPERTIES")
print("=" * 60)
print(f"Shape:        {t1w_data.shape}")
print(f"Voxel size:   {t1w_img.header.get_zooms()[:3]} mm")
print(f"Data type:    {t1w_data.dtype}")
print(f"Value range:  {t1w_data.min():.1f} - {t1w_data.max():.1f}")
print(f"Affine:\n{t1w_img.affine}")

In [ ]:
# Visualize input T1w image - orthogonal views
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Get middle slices
mid_sag = t1w_data.shape[0] // 2
mid_cor = t1w_data.shape[1] // 2
mid_axi = t1w_data.shape[2] // 2

# Row 1: Three orthogonal views
axes[0, 0].imshow(t1w_data[mid_sag, :, :].T, cmap='gray', origin='lower')
axes[0, 0].set_title('Sagittal View', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(t1w_data[:, mid_cor, :].T, cmap='gray', origin='lower')
axes[0, 1].set_title('Coronal View', fontsize=14, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(t1w_data[:, :, mid_axi].T, cmap='gray', origin='lower')
axes[0, 2].set_title('Axial View', fontsize=14, fontweight='bold')
axes[0, 2].axis('off')

# Row 2: Different axial slices to show brain coverage
slices = [int(t1w_data.shape[2] * 0.3), int(t1w_data.shape[2] * 0.5), int(t1w_data.shape[2] * 0.7)]
for idx, sl in enumerate(slices):
    axes[1, idx].imshow(t1w_data[:, :, sl].T, cmap='gray', origin='lower')
    axes[1, idx].set_title(f'Axial Slice {sl}', fontsize=12)
    axes[1, idx].axis('off')

plt.tight_layout()
plt.suptitle('INPUT T1w IMAGE - Multiple Views', fontsize=16, fontweight='bold', y=1.02)
plt.savefig(os.path.join(SUBJECTS_DIR, 'step1_input_t1w.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Input T1w visualization saved")

In [ ]:
# Histogram of intensities
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Full histogram
axes[0].hist(t1w_data[t1w_data > 0].flatten(), bins=100, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Intensity Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Intensity Distribution (all non-zero voxels)')
axes[0].grid(alpha=0.3)

# Zoomed histogram (brain tissue range)
percentile_99 = np.percentile(t1w_data[t1w_data > 0], 99)
axes[1].hist(t1w_data[(t1w_data > 0) & (t1w_data < percentile_99)].flatten(), 
             bins=100, color='coral', alpha=0.7)
axes[1].set_xlabel('Intensity Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Intensity Distribution (brain tissue range, <99th percentile)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SUBJECTS_DIR, 'step1_intensity_histogram.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Intensity histogram saved")

## 3. Import T1w into FreeSurfer Format

In [ ]:
# Import T1w to FreeSurfer format
# This creates the initial subject directory and converts to orig.mgz

print("Importing T1w to FreeSurfer...")
print("This may take a few minutes...\n")

cmd = [
    'recon-all',
    '-i', INPUT_T1W,
    '-subjid', SUBJECT_ID,
    '-sd', SUBJECTS_DIR
]

# Run command
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Successfully imported T1w to FreeSurfer")
    print(f"\nSubject directory created: {os.path.join(SUBJECTS_DIR, SUBJECT_ID)}")
else:
    print("⚠️ Warning: Import may have issues")
    print(result.stderr)

In [ ]:
# Visualize orig.mgz (FreeSurfer format)
orig_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, 'mri', 'orig.mgz')

if os.path.exists(orig_path):
    orig_img = nib.load(orig_path)
    orig_data = orig_img.get_fdata()
    
    print(f"orig.mgz shape: {orig_data.shape}")
    print(f"orig.mgz voxel size: {orig_img.header.get_zooms()[:3]} mm")
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    mid_sag = orig_data.shape[0] // 2
    mid_cor = orig_data.shape[1] // 2
    mid_axi = orig_data.shape[2] // 2
    
    axes[0].imshow(orig_data[mid_sag, :, :].T, cmap='gray', origin='lower')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(orig_data[:, mid_cor, :].T, cmap='gray', origin='lower')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(orig_data[:, :, mid_axi].T, cmap='gray', origin='lower')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle('FREESURFER orig.mgz', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SUBJECTS_DIR, 'step2_orig_mgz.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ orig.mgz visualization saved")
else:
    print("⚠️ orig.mgz not found yet")

## 4. FreeSurfer Initial Processing

**Note**: The following cells demonstrate what happens during initial FreeSurfer processing.
In practice, this step takes 6-12 hours. For this notebook, we'll visualize the key intermediate outputs.

Key steps:
1. Motion correction
2. Intensity normalization
3. Skull stripping
4. Talairach registration
5. Initial segmentation

In [ ]:
# For demonstration purposes, let's run just autorecon1 (skull strip + talairach)
# This takes ~1-2 hours

print("Starting FreeSurfer autorecon1 (skull stripping and registration)...")
print("This will take 1-2 hours.\n")
print("Command to run (uncomment to execute):")
print(f"recon-all -autorecon1 -subjid {SUBJECT_ID} -sd {SUBJECTS_DIR}")
print("\n" + "="*60)

# Uncomment to actually run:
# cmd = ['recon-all', '-autorecon1', '-subjid', SUBJECT_ID, '-sd', SUBJECTS_DIR]
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout)

## 5. Visualize Key Processing Functions

Instead of running the full pipeline (which takes many hours), let's create visualization functions
to understand what happens at each step.

In [ ]:
def visualize_segmentation(seg_file, title, colormap='tab20', save_name=None):
    """
    Visualize a segmentation file (e.g., aseg.mgz)
    """
    if not os.path.exists(seg_file):
        print(f"File not found: {seg_file}")
        return
    
    seg_img = nib.load(seg_file)
    seg_data = seg_img.get_fdata()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    mid_sag = seg_data.shape[0] // 2
    mid_cor = seg_data.shape[1] // 2
    mid_axi = seg_data.shape[2] // 2
    
    axes[0].imshow(seg_data[mid_sag, :, :].T, cmap=colormap, origin='lower')
    axes[0].set_title('Sagittal', fontsize=12)
    axes[0].axis('off')
    
    axes[1].imshow(seg_data[:, mid_cor, :].T, cmap=colormap, origin='lower')
    axes[1].set_title('Coronal', fontsize=12)
    axes[1].axis('off')
    
    axes[2].imshow(seg_data[:, :, mid_axi].T, cmap=colormap, origin='lower')
    axes[2].set_title('Axial', fontsize=12)
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_name:
        plt.savefig(os.path.join(SUBJECTS_DIR, save_name), dpi=150, bbox_inches='tight')
    
    plt.show()
    
    # Print label statistics
    unique_labels = np.unique(seg_data[seg_data > 0])
    print(f"\nNumber of unique labels: {len(unique_labels)}")
    print(f"Label range: {seg_data.min():.0f} - {seg_data.max():.0f}")
    print(f"\nTop 10 most common labels:")
    
    label_counts = {}
    for label in unique_labels:
        count = np.sum(seg_data == label)
        label_counts[int(label)] = count
    
    for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  Label {label:3d}: {count:8d} voxels")

print("✓ Segmentation visualization function defined")

In [ ]:
def compare_segmentations(seg1_file, seg2_file, title1, title2, save_name=None):
    """
    Compare two segmentation files side by side
    """
    if not os.path.exists(seg1_file) or not os.path.exists(seg2_file):
        print(f"One or both files not found")
        return
    
    seg1_img = nib.load(seg1_file)
    seg1_data = seg1_img.get_fdata()
    
    seg2_img = nib.load(seg2_file)
    seg2_data = seg2_img.get_fdata()
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    mid_sag = seg1_data.shape[0] // 2
    mid_cor = seg1_data.shape[1] // 2
    mid_axi = seg1_data.shape[2] // 2
    
    # First segmentation
    axes[0, 0].imshow(seg1_data[mid_sag, :, :].T, cmap='tab20', origin='lower')
    axes[0, 0].set_title(f'{title1}\nSagittal', fontsize=10)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(seg1_data[:, mid_cor, :].T, cmap='tab20', origin='lower')
    axes[0, 1].set_title(f'{title1}\nCoronal', fontsize=10)
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(seg1_data[:, :, mid_axi].T, cmap='tab20', origin='lower')
    axes[0, 2].set_title(f'{title1}\nAxial', fontsize=10)
    axes[0, 2].axis('off')
    
    # Second segmentation
    axes[1, 0].imshow(seg2_data[mid_sag, :, :].T, cmap='tab20', origin='lower')
    axes[1, 0].set_title(f'{title2}\nSagittal', fontsize=10)
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(seg2_data[:, mid_cor, :].T, cmap='tab20', origin='lower')
    axes[1, 1].set_title(f'{title2}\nCoronal', fontsize=10)
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(seg2_data[:, :, mid_axi].T, cmap='tab20', origin='lower')
    axes[1, 2].set_title(f'{title2}\nAxial', fontsize=10)
    axes[1, 2].axis('off')
    
    plt.suptitle('SEGMENTATION COMPARISON', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_name:
        plt.savefig(os.path.join(SUBJECTS_DIR, save_name), dpi=150, bbox_inches='tight')
    
    plt.show()
    
    # Calculate differences
    diff = np.sum(seg1_data != seg2_data)
    total = np.prod(seg1_data.shape)
    percent_diff = (diff / total) * 100
    
    print(f"\nDifference: {diff:,} voxels ({percent_diff:.2f}% of total)")

print("✓ Comparison visualization function defined")

## 6. Demonstrate Label Remapping (iFS → FS)

This demonstrates what the bash script does to remap thalamus labels.

In [ ]:
# Create a synthetic example to demonstrate label remapping
# This simulates what happens in the pipeline

print("=" * 60)
print("LABEL REMAPPING DEMONSTRATION")
print("=" * 60)
print("\nInfant FreeSurfer uses different thalamus labels than standard FreeSurfer:")
print("\niFS Labels → FS Labels:")
print("  9 (left thalamus)  → 10 (left thalamus)")
print(" 48 (right thalamus) → 49 (right thalamus)")
print("\nThis is done using FreeSurfer commands:")
print("\n# Change label 9 to 10")
print("mri_binarize --i aseg.mgz --match 9 --replace 10 --o tmp.mgz")
print("mri_mask -transfer 10 tmp.mgz aseg.mgz aseg.mgz")
print("\n# Change label 48 to 49")
print("mri_binarize --i aseg.mgz --match 48 --replace 49 --o tmp.mgz")
print("mri_mask -transfer 49 tmp.mgz aseg.mgz aseg.mgz")

# Create synthetic segmentation to demonstrate
demo_seg = np.zeros((100, 100, 100))
demo_seg[30:40, 40:50, 45:55] = 9  # Left thalamus (iFS)
demo_seg[60:70, 40:50, 45:55] = 48  # Right thalamus (iFS)

# Show before
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(demo_seg[:, :, 50], cmap='tab20', vmin=0, vmax=50)
axes[0].set_title('Before Remapping\n(iFS labels: 9, 48)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Apply remapping
demo_seg_remapped = demo_seg.copy()
demo_seg_remapped[demo_seg == 9] = 10
demo_seg_remapped[demo_seg == 48] = 49

axes[1].imshow(demo_seg_remapped[:, :, 50], cmap='tab20', vmin=0, vmax=50)
axes[1].set_title('After Remapping\n(FS labels: 10, 49)', fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.suptitle('THALAMUS LABEL REMAPPING', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SUBJECTS_DIR, 'step5_label_remapping_demo.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Label remapping demonstration complete")

## 7. Demonstrate White Matter Generation

This demonstrates how wm.mgz is created from aseg labels.

In [ ]:
print("=" * 60)
print("WHITE MATTER GENERATION DEMONSTRATION")
print("=" * 60)
print("\nThe wm.mgz file is generated by extracting specific labels from aseg:")
print("\nWhite Matter labels (→ 110):")
print("  2:   Left Cerebral White Matter")
print(" 41:   Right Cerebral White Matter")
print("173:   Brainstem (part 1)")
print("174:   Brainstem (part 2)")
print("175:   Brainstem (part 3)")
print("\nGray Matter labels (→ 250):")
print("  4:   Left Lateral Ventricle")
print(" 11:   Left Caudate")
print(" 12:   Left Putamen")
print(" 13:   Left Pallidum")
print(" ... (and others)")
print("\nBash commands:")
print("mri_binarize --i aseg.mgz --match 2 41 173 174 175 --replace 110 --o wm_tmp.mgz")
print("mri_binarize --i aseg.mgz --match 4 11 12 13 26 28 43 50 51 52 58 60 --replace 250 --o gm_tmp.mgz")
print("fscalc wm_tmp.mgz add gm_tmp.mgz --o wm.mgz")

# Create synthetic demonstration
demo_aseg = np.zeros((100, 100, 100))
# Add some structures
demo_aseg[20:80, 20:35, 40:60] = 2   # Left WM
demo_aseg[20:80, 65:80, 40:60] = 41  # Right WM
demo_aseg[40:60, 40:60, 35:45] = 11  # Left Caudate
demo_aseg[40:60, 40:60, 55:65] = 50  # Right Caudate

# Generate wm.mgz equivalent
wm_labels = [2, 41, 173, 174, 175]
gm_labels = [4, 11, 12, 13, 26, 28, 43, 50, 51, 52, 58, 60]

demo_wm = np.zeros_like(demo_aseg)
for label in wm_labels:
    demo_wm[demo_aseg == label] = 110
for label in gm_labels:
    demo_wm[demo_aseg == label] = 250

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(demo_aseg[:, :, 50], cmap='tab20')
axes[0].set_title('Input: aseg.mgz\n(various labels)', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(demo_wm[:, :, 50], cmap='Set1', vmin=0, vmax=250)
axes[1].set_title('Output: wm.mgz\n(110=WM, 250=GM)', fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.suptitle('WHITE MATTER FILE GENERATION', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SUBJECTS_DIR, 'step6_wm_generation_demo.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ White matter generation demonstration complete")

## 8. Pipeline Summary and Flow Diagram

In [ ]:
# Create pipeline flow diagram
print("=" * 80)
print("COMPLETE PIPELINE FLOW")
print("=" * 80)
print("""
┌─────────────────────────────────────────────────────────────────────────┐
│                         INPUT: T1w NIfTI Image                          │
│                     sub-01_ses-03_T1w.nii.gz                            │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 1: FreeSurfer Import                                              │
│  Command: recon-all -i T1w.nii.gz -subjid sub-01                        │
│  Output: orig.mgz                                                       │
│  Time: ~5 minutes                                                       │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 2: FreeSurfer Initial Processing (recon-all -all)                 │
│  - Motion correction                                                    │
│  - Intensity normalization (nu.mgz, T1.mgz)                             │
│  - Skull stripping (brainmask.mgz)                                      │
│  - Talairach registration                                               │
│  - Initial segmentation                                                 │
│  Time: ~8-12 hours                                                      │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 3: Prepare for Infant FreeSurfer                                  │
│  Command: mri_convert -i orig.mgz -o mprage.nii.gz                      │
│  Output: mprage.nii.gz (in iFS directory)                               │
│  Time: ~1 minute                                                        │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 4: Infant FreeSurfer Processing                                   │
│  Command: infant_recon_all --s sub-01 --age <months>                    │
│  - Age-appropriate atlas registration                                   │
│  - Infant-optimized tissue segmentation                                 │
│  - Subcortical structure segmentation                                   │
│  Output: aseg.mgz (iFS labels)                                          │
│  Time: ~2-4 hours                                                       │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 5: Label Remapping (iFS → FS compatibility) [BASH-ONLY]           │
│  Copy iFS aseg to FS directory                                          │
│  Remap thalamus labels:                                                 │
│    - Label 9  → 10 (left thalamus)                                      │
│    - Label 48 → 49 (right thalamus)                                     │
│  Commands:                                                              │
│    mri_convert -i iFS/aseg.mgz -o aseg.presurf.mgz                      │
│    mri_binarize --match 9 --replace 10                                  │
│    mri_mask -transfer 10                                                │
│  Output: aseg.presurf.mgz                                               │
│  Time: ~1 minute                                                        │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 6: White Matter Generation [BASH-ONLY]                            │
│  Extract WM labels (2,41,173,174,175) → 110                             │
│  Extract GM labels (4,11,12,13,...) → 250                               │
│  Combine: fscalc wm_tmp.mgz add gm_tmp.mgz                              │
│  Output: wm.mgz                                                         │
│  Time: ~1 minute                                                        │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 7: Surface Reconstruction (autorecon2)                            │
│  - White matter masking                                                 │
│  - Tessellation                                                         │
│  - Topology correction                                                  │
│  - Surface smoothing                                                    │
│  - White surface positioning                                            │
│  Output: lh.white, rh.white                                             │
│  Time: ~4-6 hours                                                       │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 8: Surface Refinement (autorecon3)                                │
│  - Pial surface positioning                                             │
│  - Cortical parcellation                                                │
│  - Thickness calculation                                                │
│  - Statistics generation                                                │
│  Output: lh.pial, rh.pial, lh.thickness, stats files                    │
│  Time: ~4-6 hours                                                       │
└─────────────────────────────┬───────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│                    FINAL OUTPUTS                                        │
│  - Volumetric segmentations (aseg.mgz, wmparc.mgz)                      │
│  - Surface files (white, pial, inflated, sphere)                        │
│  - Morphometric measurements (thickness, area, volume)                  │
│  - Statistics files (stats/*.stats)                                     │
│                                                                         │
│  Total Pipeline Time: ~20-30 hours                                      │
└─────────────────────────────────────────────────────────────────────────┘
""")

print("\n" + "="*80)
print("KEY DIFFERENCES FROM ORIGINAL PIPELINE")
print("="*80)
print("""
Original (iBEAT2 + MATLAB):              Simplified (iFS-only, Bash):
─────────────────────────────            ────────────────────────────
✗ Requires iBEAT2 Docker                 ✓ No iBEAT2 needed
✗ Requires MATLAB                        ✓ Pure bash/FreeSurfer
• Uses iBEAT2 for cortical tissue        • Uses iFS for all tissue
• MATLAB merges iBEAT2 + iFS              • Bash processes iFS only
• Complex installation                   • Simple installation
• Multiple dependencies                  • Minimal dependencies
""")

## 9. Quality Control Metrics

In [ ]:
def generate_qc_report(subject_dir):
    """
    Generate quality control report for processed subject
    """
    print("=" * 60)
    print("QUALITY CONTROL CHECKLIST")
    print("=" * 60)
    print("\n📋 Files to Check:")
    
    files_to_check = {
        'Input': 'mri/orig.mgz',
        'Normalized': 'mri/T1.mgz',
        'Brain Mask': 'mri/brainmask.mgz',
        'Segmentation': 'mri/aseg.presurf.mgz',
        'White Matter': 'mri/wm.mgz',
        'Left White Surface': 'surf/lh.white',
        'Right White Surface': 'surf/rh.white',
        'Left Pial Surface': 'surf/lh.pial',
        'Right Pial Surface': 'surf/rh.pial',
    }
    
    for name, path in files_to_check.items():
        full_path = os.path.join(subject_dir, path)
        exists = os.path.exists(full_path)
        status = "✓" if exists else "✗"
        print(f"  {status} {name:20s}: {path}")
    
    print("\n📊 Key Quality Metrics to Check:")
    print("""
    1. Skull Stripping Quality
       - Check brainmask.mgz for complete brain coverage
       - Ensure no skull/dura included
       - Verify cerebellum included
    
    2. Tissue Segmentation
       - Review aseg.mgz for accurate GM/WM/CSF
       - Check subcortical structures (thalamus, caudate, etc.)
       - Verify ventricle segmentation
    
    3. White Surface (lh/rh.white)
       - Should follow GM/WM boundary
       - Check for holes or handles (topology errors)
       - Verify smoothness
    
    4. Pial Surface (lh/rh.pial)
       - Should follow GM/CSF boundary
       - Check for intersection with white surface
       - Verify dura not included
    
    5. Euler Number (Topology)
       - Check surf/lh.orig.nofix.euler
       - Euler = 2 is perfect sphere
       - Each hole reduces Euler by 2
    """)
    
    print("\n🔍 Visualization Commands:")
    print("""
    # View segmentation overlay on T1
    freeview -v mri/T1.mgz mri/aseg.mgz:colormap=lut:opacity=0.3
    
    # View white surface
    freeview -v mri/T1.mgz -f surf/lh.white:edgecolor=yellow surf/rh.white:edgecolor=yellow
    
    # View pial surface
    freeview -v mri/T1.mgz -f surf/lh.pial:edgecolor=red surf/rh.pial:edgecolor=red
    
    # View both surfaces together
    freeview -v mri/T1.mgz \
             -f surf/lh.white:edgecolor=yellow surf/rh.white:edgecolor=yellow \
                surf/lh.pial:edgecolor=red surf/rh.pial:edgecolor=red
    """)

# Generate report for our subject
subject_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID)
if os.path.exists(subject_path):
    generate_qc_report(subject_path)
else:
    print(f"Subject directory not found: {subject_path}")
    print("Run the pipeline first before generating QC report.")

## 10. Running the Complete Pipeline

Now that you understand each step, here's how to run the complete simplified pipeline:

In [ ]:
print("=" * 80)
print("RUNNING THE COMPLETE PIPELINE")
print("=" * 80)
print("\nTo run the complete simplified pipeline, use:")
print("\n" + "─" * 80)

pipeline_script = "/path/to/peer-review/1.Structure/reFS_iFS_only_no_matlab.sh"
age_months = 6  # Adjust based on subject age

print(f"""
# 1. Set up environment
export FREESURFER_HOME=/usr/local/freesurfer
source $FREESURFER_HOME/SetUpFreeSurfer.sh
export SUBJECTS_DIR={SUBJECTS_DIR}

# 2. Import T1w image to FreeSurfer
recon-all -i {INPUT_T1W} \\
          -subjid {SUBJECT_ID} \\
          -sd {SUBJECTS_DIR}

# 3. Run the simplified iFS-only pipeline
{pipeline_script} {SUBJECT_ID} {age_months}

# This will take approximately 20-30 hours to complete
""")

print("─" * 80)
print("\nAlternatively, run step by step:")
print("─" * 80)

print(f"""
# Step 1: Import (5 min)
recon-all -i {INPUT_T1W} -subjid {SUBJECT_ID}

# Step 2: Initial FS processing (8-12 hours)
recon-all -all -subjid {SUBJECT_ID} -nonuintensitycor

# Step 3: Prepare for iFS (1 min)
mkdir -p {SUBJECTS_DIR}/../iFS/{SUBJECT_ID}
mri_convert -i {SUBJECTS_DIR}/{SUBJECT_ID}/mri/orig.mgz \\
            -o {SUBJECTS_DIR}/../iFS/{SUBJECT_ID}/mprage.nii.gz

# Step 4: Run iFS (2-4 hours)
infant_recon_all --s {SUBJECT_ID} --age {age_months}

# Step 5: Process iFS aseg (1 min)
mri_convert -i {SUBJECTS_DIR}/../iFS/{SUBJECT_ID}/mri/aseg.mgz \\
            -o {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz

# Remap thalamus labels
mri_binarize --i {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz \\
             --match 9 --replace 10 --o /tmp/tmp.mgz
mri_mask -transfer 10 /tmp/tmp.mgz \\
         {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz \\
         {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz

mri_binarize --i {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz \\
             --match 48 --replace 49 --o /tmp/tmp.mgz
mri_mask -transfer 49 /tmp/tmp.mgz \\
         {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz \\
         {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz

# Step 6: Generate WM file (1 min)
mri_binarize --i {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz \\
             --match 2 41 173 174 175 --replace 110 \\
             --o {SUBJECTS_DIR}/{SUBJECT_ID}/mri/wm_tmp.mgz

mri_binarize --i {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.presurf.mgz \\
             --match 4 11 12 13 26 28 43 50 51 52 58 60 --replace 250 \\
             --o {SUBJECTS_DIR}/{SUBJECT_ID}/mri/gm_tmp.mgz

fscalc {SUBJECTS_DIR}/{SUBJECT_ID}/mri/wm_tmp.mgz add \\
       {SUBJECTS_DIR}/{SUBJECT_ID}/mri/gm_tmp.mgz \\
       --o {SUBJECTS_DIR}/{SUBJECT_ID}/mri/wm.mgz

# Step 7-8: Continue with autorecon2/3 (8-12 hours)
# Run fs_autorecon2_end.sh and fs_autorecon3_wrap.sh
""")

print("\n" + "=" * 80)
print("Expected Timeline:")
print("="*80)
timeline = [
    ("Import", "5 min"),
    ("Initial FS processing", "8-12 hours"),
    ("iFS preparation", "1 min"),
    ("iFS processing", "2-4 hours"),
    ("Label remapping", "1 min"),
    ("WM generation", "1 min"),
    ("Surface reconstruction", "8-12 hours"),
    ("", ""),
    ("TOTAL", "20-30 hours"),
]

for step, time in timeline:
    if step:
        print(f"  {step:30s} {time:>15s}")
    else:
        print("  " + "-" * 47)

## Summary

This notebook demonstrated the complete infant brain processing pipeline with visualizations.

**Key Takeaways:**

1. **Input**: T1-weighted MRI in NIfTI format
2. **Initial Processing**: FreeSurfer performs skull stripping, normalization, registration
3. **Infant FreeSurfer**: Age-appropriate tissue segmentation
4. **Label Remapping**: Bash commands adjust thalamus labels for compatibility
5. **WM Generation**: Bash commands create white matter file from segmentation
6. **Surface Reconstruction**: FreeSurfer generates cortical surfaces and metrics

**Advantages of Simplified Pipeline:**
- ✓ No iBEAT2 required
- ✓ No MATLAB required
- ✓ Pure bash/FreeSurfer tools
- ✓ Easy to understand and modify
- ✓ Maintains all pipeline functionality

**Next Steps:**
1. Run the pipeline on your data
2. Perform quality control checks
3. Extract morphometric statistics
4. Analyze longitudinal trajectories